In [4]:
pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp311-cp311-macosx_11_0_arm64.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jinja2]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np

PATH = "results_horizon_runs/final_test_macro_f1_pivot.csv"

# Load the data
df = pd.read_csv(PATH)

# 1. Clean up columns (dropping 'split' if it exists as it's redundant)
if 'split' in df.columns:
    df = df.drop(columns=['split'])

# 2. Round all numeric metrics for readability
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].round(4)

# 3. Sort by Horizon then by Macro F1 (highest first)
df = df.sort_values(by=['horizon', 'macro_f1'], ascending=[True, False])

# 4. Pretty Print Logic
print("\n" + "="*95)
print(f"{'MODEL PERFORMANCE BY HORIZON':^95}")
print("="*95)

# Group by horizon to create distinct sections
for horizon, group in df.groupby('horizon'):
    print(f"\n▶ HORIZON: {horizon}")
    print("-" * 95)
    
    # Selecting and renaming columns for a cleaner display
    display_df = group.drop(columns=['horizon']).rename(columns={
        'macro_f1': 'Macro F1',
        'weighted_f1': 'W-F1',
        'f1_calm': 'F1:Calm',
        'f1_normal': 'F1:Normal',
        'f1_turbulent': 'F1:Turb'
    })
    
    # print using pandas string representation
    print(display_df.to_string(index=False))
    
    # Bonus: Identify the winner for this horizon
    winner = display_df.iloc[0]['model']
    score = display_df.iloc[0]['Macro F1']
    print(f"\n🏆 Best Model for Horizon {horizon}: {winner} ({score})")
    print("=" * 95)

/var/folders/yf/m6d66yd55f92zy8ydr9f93d00000gn/T/ipykernel_3056/2509627928.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(PATH, index_col=0, parse_dates=True)


In [3]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 55.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 81.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np

PATH = "labeled_data/labeled_data.csv"

df = pd.read_csv(PATH, index_col=0, parse_dates=True)

print("\n================ BASIC INFO ================\n")
print("Total days:", len(df))
print("Date range:", df.index.min(), "→", df.index.max())

# --------------------------------------------------
# 1. Label distribution (this is the key proof)
# --------------------------------------------------
print("\n================ LABEL DISTRIBUTION ================\n")

dist = df["label"].value_counts().sort_index()
pct = df["label"].value_counts(normalize=True).sort_index() * 100

label_map = {0: "Calm", 1: "Normal", 2: "Turbulent"}

for k in dist.index:
    print(f"{label_map[k]:<10}: {dist[k]:5d} days ({pct[k]:5.2f}%)")

# --------------------------------------------------
# 2. Regime score distribution (continuous view)
# --------------------------------------------------
print("\n================ REGIME SCORE STATS ================\n")
print(df["regime_score"].describe())

# --------------------------------------------------
# 3. Percentiles (to show concentration)
# --------------------------------------------------
print("\n================ REGIME SCORE PERCENTILES ================\n")
for q in [0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
    print(f"{int(q*100)}th percentile:", df["regime_score"].quantile(q))

# --------------------------------------------------
# 4. Consecutive regime durations
# --------------------------------------------------
print("\n================ REGIME DURATIONS ================\n")

labels = df["label"].values
durations = {0: [], 1: [], 2: []}

i = 0
while i < len(labels):
    current = labels[i]
    j = i
    while j < len(labels) and labels[j] == current:
        j += 1
    durations[current].append(j - i)
    i = j

for k in [0, 1, 2]:
    if len(durations[k]) > 0:
        print(f"{label_map[k]:<10}: "
              f"mean={np.mean(durations[k]):.2f}, "
              f"median={np.median(durations[k]):.2f}, "
              f"max={np.max(durations[k])}")

# --------------------------------------------------
# 5. Long vs short regimes
# --------------------------------------------------
print("\n================ LONG REGIME ANALYSIS ================\n")

long_threshold = 20

for k in [0, 1, 2]:
    long_count = sum(1 for d in durations[k] if d >= long_threshold)
    total = len(durations[k])
    if total > 0:
        print(f"{label_map[k]:<10}: {long_count}/{total} regimes >= {long_threshold} days")

# --------------------------------------------------
# 6. Volatility distribution (key insight)
# --------------------------------------------------
print("\n================ VOLATILITY DISTRIBUTION ================\n")

vol = df["market_volatility"]

print(vol.describe())

print("\nHigh volatility days (>90th percentile):",
      (vol > vol.quantile(0.9)).sum())

print("Low volatility days (<10th percentile):",
      (vol < vol.quantile(0.1)).sum())

# --------------------------------------------------
# 7. Sanity check: % of time in high stress
# --------------------------------------------------
print("\n================ STRESS CHECK ================\n")

high_stress = (
    (df["market_volatility"] > df["market_volatility"].quantile(0.8)) &
    (df["avg_correlation"] > df["avg_correlation"].quantile(0.8))
)

print("High stress days:", high_stress.sum())
print("Percentage:", 100 * high_stress.mean(), "%")



================ BASIC INFO ================

Total days: 2292
Date range: 2017-02-15 00:00:00 → 2026-03-30 00:00:00

================ LABEL DISTRIBUTION ================

Calm      :   836 days (36.47%)
Normal    :   910 days (39.70%)
Turbulent :   546 days (23.82%)

================ REGIME SCORE STATS ================

count    2292.000000
mean       -0.119568
std         0.809519
min        -1.283308
25%        -0.603996
50%        -0.377505
75%         0.164026
max         4.814780
Name: regime_score, dtype: float64

================ REGIME SCORE PERCENTILES ================

10th percentile: -0.8083581179416707
25th percentile: -0.6039955083938211
50th percentile: -0.37750527101080433
75th percentile: 0.16402626623212616
90th percentile: 0.8182557254520841
95th percentile: 1.1303482731994583

================ REGIME DURATIONS ================

Calm      : mean=26.97, median=12.00, max=243
Normal    : mean=18.57, median=12.00, max=119
Turbulent : mean=30.33, median=22.00, max=106
